# Class Conversion

Code snippets to quickly convert across the main classes provided by HiCONA, highlighting which information is lost at each step. This is a non-exaustive list of possible cases; here are the most common situations, for more complex corner cases always use class constructors.

In [9]:
import os
import cooler as co
import hicona as hi
import polars as pl


# Initial setup
COOLER_PATH = "../data/IMR90_100kb_chr17_balanced.cool"
MOCK_COOLER = "some_cooler_path.cool"
REGION = "chr17:0-10000000"
cooler = hi.HiconaCooler(COOLER_PATH)

# HiconaCooler -> PixelTable
# - keeps all bins
# - loses pixels outside of the region
table = cooler.get_pixel_table(REGION)  

# PixelTable -> HiconaCooler
# - keeps all bins
# - pixels outside of the subset region are not recovered
co.create_cooler(
    MOCK_COOLER, 
    bins=table.bins.get_dataframe(dtype="pandas"), 
    pixels=table.get_chunks(dtype="pandas")
)
new_cooler = hi.HiconaCooler(MOCK_COOLER)
os.remove(MOCK_COOLER)  # Comment to keep the new cooler

# PixelTable -> HiconaGraph
# - loses bins outside of the region
# - loses pixels outside the region (if any)
graph = table.get_graph(REGION)

# HiconaGraph -> PixelTable
# - keeps all initial bins
# - only keep pixels to which the graph was subsetted
new_table = hi.PixelTable(
    bins=table.bins.get_dataframe().join(pl.concat(graph.get_bins()), on=table.bins.col_names, how="left"), 
    pixels=graph.get_pixels()
)
# or
# - keeps all initial bins
# - keep all table pixels (which might still be a subset of the cooler), more memory intensive
new_table = hi.PixelTable(
    bins=table.bins.get_dataframe().join(pl.concat(graph.get_bins()), on=table.bins.col_names, how="left"), 
    pixels=pl.concat(graph.get_pixels()).join(pl.concat(graph.get_pixels()), on=table.col_names, how="left")
)
